In [1]:
!pip install concrete-python

In [4]:
import numpy as np
from concrete import fhe


# ============================================================================
# Task 4: FHE Circuits for Mean Computation
# ============================================================================

def create_mean_circuit_6():
    """
    Create FHE circuit for integer mean of 6 values.
    Uses integer division, which truncates decimal places.
    """
    @fhe.compiler({"x": "encrypted"})
    def mean_of_6(x):
        total = np.sum(x)
        return total // 6

    # Inputset must cover the range of values we want to test
    inputset = [np.array([i, i+10, i+20, i+30, i+40, i+50]) for i in range(0, 600, 50)]
    return mean_of_6.compile(inputset)


def create_mean_circuit_6_decimal():
    """
    Create FHE circuit for mean with 2 decimal precision.
    Returns result scaled by 100 (e.g., 12.34 → 1234).
    """
    @fhe.compiler({"x": "encrypted"})
    def mean_of_6_decimal(x):
        total = np.sum(x)
        return (total * 100) // 6  # Scale by 100 for 2 decimals

    inputset = [np.array([i, i+1, i+2, i+3, i+4, i+5]) for i in range(0, 100, 10)]
    return mean_of_6_decimal.compile(inputset)


# ============================================================================
# Circuit for 7 integers (Answer to Question a)
# ============================================================================

def create_mean_circuit_7():
    """
    Create FHE circuit for integer mean of 7 values.
    Demonstrates how to adapt for different list sizes.
    """
    @fhe.compiler({"x": "encrypted"})
    def mean_of_7(x):
        total = np.sum(x)
        return total // 7

    inputset = [np.array([i, i+10, i+20, i+30, i+40, i+50, i+60]) for i in range(0, 600, 50)]
    return mean_of_7.compile(inputset)


# ============================================================================
# TEST CASES
# ============================================================================

def run_test(circuit, test_name, test_data, expected, is_decimal=False):
    """Helper function to run a single test."""
    try:
        result = circuit.encrypt_run_decrypt(test_data)
        passed = result == expected

        print(f"{test_name}")
        print(f"  Input: {test_data}")
        if is_decimal:
            print(f"  Expected: {expected/100:.2f} (scaled: {expected})")
            print(f"  Result:   {result/100:.2f} (scaled: {result})")
        else:
            print(f"  Expected: {expected}")
            print(f"  Result:   {result}")
        print(f"  Status: {'✓ PASS' if passed else '✗ FAIL'}\n")
        return passed
    except Exception as e:
        print(f"{test_name}")
        print(f"  ✗ FAIL - Error: {e}\n")
        return False


def run_tests():
    print("\n" + "="*70)
    print("    Task 4: Fully Homomorphic Encryption - Mean Computation")
    print("="*70 + "\n")

    passed = 0
    total = 11  # Updated total

    # ========================================================================
    # Part 1: Integer Mean of 6 Values
    # ========================================================================
    print("PART 1: Integer Mean of 6 Encrypted Values\n")
    print("Compiling circuit...")
    circuit_int = create_mean_circuit_6()
    print("✓ Compiled\n")

    # Test Case Selection Rationale:
    # 1.1: Simple arithmetic - verifies basic addition and division work correctly
    # 1.2: All same values - tests homogeneity, ensures division by 6 is exact
    # 1.3: Sequential small values - tests lower bound, verifies integer truncation
    # 1.4: Large values - tests upper bound of inputset, ensures no overflow
    # 1.5: Zero included - edge case, tests handling of zero in encrypted form

    tests_int = [
        ("Test 1.1: Simple sequence",
         np.array([10, 20, 30, 40, 50, 60]), 35,
         "Basic arithmetic correctness"),

        ("Test 1.2: All same values",
         np.array([5, 5, 5, 5, 5, 5]), 5,
         "Homogeneous data, exact division"),

        ("Test 1.3: Sequential values",
         np.array([1, 2, 3, 4, 5, 6]), 3,
         "Integer truncation (21/6 = 3.5 → 3)"),

        ("Test 1.4: Large values",
         np.array([100, 200, 300, 400, 500, 600]), 350,
         "Upper bound test, no overflow"),

        ("Test 1.5: Zero included",
         np.array([0, 10, 20, 30, 40, 50]), 25,
         "Edge case with zero value"),
    ]

    for name, data, expected, rationale in tests_int:
        print(f"  Rationale: {rationale}")
        if run_test(circuit_int, name, data, expected):
            passed += 1

    # ========================================================================
    # Part 2: Decimal Mean with 2 Decimal Precision
    # ========================================================================
    print("="*70)
    print("PART 2: Mean with 2 Decimal Precision\n")
    print("Compiling circuit...")
    circuit_dec = create_mean_circuit_6_decimal()
    print("✓ Compiled\n")

    # Test Case Selection Rationale:
    # 2.1: Half value (x.5) - tests precision preservation for .50
    # 2.2: Clean division - verifies integer results still work correctly
    # 2.3: Repeating decimal - tests truncation at 2 decimal places
    # 2.4: All same values - baseline for scaled computation

    tests_dec = [
        ("Test 2.1: Decimal .50 result",
         np.array([10, 11, 12, 13, 14, 15]), 1250,
         "Tests .50 precision (12.50)"),

        ("Test 2.2: Clean integer result",
         np.array([6, 12, 18, 24, 30, 36]), 2100,
         "Integer result in decimal form (21.00)"),

        ("Test 2.3: Repeating decimal",
         np.array([1, 2, 3, 4, 5, 6]), 350,
         "Truncates .333... to .50 (3.50)"),

        ("Test 2.4: Homogeneous data",
         np.array([1, 1, 1, 1, 1, 1]), 100,
         "Baseline scaled precision (1.00)"),
    ]

    for name, data, expected, rationale in tests_dec:
        print(f"  Rationale: {rationale}")
        if run_test(circuit_dec, name, data, expected, is_decimal=True):
            passed += 1

    # ========================================================================
    # Part 3: Bonus - Mean of 7 Values (Answer to Question a)
    # ========================================================================
    print("="*70)
    print("PART 3: Mean of 7 Encrypted Values (Extension)\n")
    print("Compiling circuit for 7 values...")
    circuit_7 = create_mean_circuit_7()
    print("✓ Compiled\n")

    # Demonstrates how to handle different list sizes
    tests_7 = [
        ("Test 3.1: Seven values",
         np.array([10, 20, 30, 40, 50, 60, 70]), 40,
         "Demonstrates adaptation to 7 values"),

        ("Test 3.2: Seven same values",
         np.array([7, 7, 7, 7, 7, 7, 7]), 7,
         "Exact division by 7"),
    ]

    for name, data, expected, rationale in tests_7:
        print(f"  Rationale: {rationale}")
        if run_test(circuit_7, name, data, expected):
            passed += 1

    # ========================================================================
    # Summary
    # ========================================================================
    print("="*70)
    print("TEST COVERAGE SUMMARY:")
    print(f"  Tests passed: {passed}/{total}")
    print("\nCoverage Dimensions:")
    print("  ✓ Arithmetic correctness (basic operations)")
    print("  ✓ Edge cases (zero, homogeneous data)")
    print("  ✓ Boundary values (small and large numbers)")
    print("  ✓ Precision handling (integer truncation, decimal scaling)")
    print("  ✓ Scalability (extension to 7 values)")
    print("="*70 + "\n")

    # ========================================================================
    # Demonstration: Separate Encryption Steps
    # ========================================================================
    print("DEMONSTRATION: Separate Encrypt/Compute/Decrypt Steps\n")
    demo_data = np.array([10, 20, 30, 40, 50, 60])
    print(f"Input: {demo_data}")

    encrypted_input = circuit_int.encrypt(demo_data)
    print("✓ Encrypted")

    encrypted_output = circuit_int.run(encrypted_input)
    print("✓ Computed on encrypted data")

    result = circuit_int.decrypt(encrypted_output)
    print(f"✓ Decrypted: {result}")
    print(f"  Expected: {sum(demo_data) // 6}")
    print(f"  {'✓ CORRECT' if result == 35 else '✗ INCORRECT'}\n")

    # ========================================================================
    # Limitations Demonstration
    # ========================================================================
    print("="*70)
    print("LIMITATIONS DEMONSTRATION:")
    print("="*70 + "\n")

    print("1. Fixed Input Size:")
    print("   - Circuit compiled for 6 values cannot process 5 or 7")
    print("   - Each size requires separate compilation\n")

    print("2. Integer Division Loss:")
    print("   - Mean of [1,2,3,4,5,6] = 3 (not 3.5)")
    print("   - Solution: Use scaled version (Part 2)\n")

    print("3. Limited Decimal Precision:")
    print("   - Scaled version only provides 2 decimal places")
    print("   - Mean of [1,2,3] = 3.50 (actual: 3.333...)\n")

    print("4. Input Range Restriction:")
    print("   - Values must be within inputset range")
    print("   - Testing 1000+ would fail without recompilation\n")

    print("5. Compilation Overhead:")
    print("   - Each circuit compilation takes seconds")
    print("   - Not suitable for dynamic applications\n")

    print("="*70 + "\n")


if __name__ == "__main__":
    run_tests()


    Task 4: Fully Homomorphic Encryption - Mean Computation

PART 1: Integer Mean of 6 Encrypted Values

Compiling circuit...
✓ Compiled

  Rationale: Basic arithmetic correctness
Test 1.1: Simple sequence
  Input: [10 20 30 40 50 60]
  Expected: 35
  Result:   35
  Status: ✓ PASS

  Rationale: Homogeneous data, exact division
Test 1.2: All same values
  Input: [5 5 5 5 5 5]
  Expected: 5
  Result:   5
  Status: ✓ PASS

  Rationale: Integer truncation (21/6 = 3.5 → 3)
Test 1.3: Sequential values
  Input: [1 2 3 4 5 6]
  Expected: 3
  Result:   3
  Status: ✓ PASS

  Rationale: Upper bound test, no overflow
Test 1.4: Large values
  Input: [100 200 300 400 500 600]
  Expected: 350
  Result:   350
  Status: ✓ PASS

  Rationale: Edge case with zero value
Test 1.5: Zero included
  Input: [ 0 10 20 30 40 50]
  Expected: 25
  Result:   25
  Status: ✓ PASS

PART 2: Mean with 2 Decimal Precision

Compiling circuit...
✓ Compiled

  Rationale: Tests .50 precision (12.50)
Test 2.1: Decimal .50 res